# Solutions · 04 · The root locus

Worked solutions to the exercises of [notebook 04](../notebooks/04_the_root_locus.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere englincontrol is missing): install it from GitHub
    import englincontrol
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/englincontrol"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from englincontrol import systems as sy, stability as st, polynomials as pl
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Draw the root locus of 1/(s (s + 4)(s^2 + 4s + 20)) by the rules first (asymptotes, breakaway, departure angles), then compute it. At what gain does it cross the imaginary axis, and does Routh agree?

In [2]:
G = sy.tf([1.0], pl.mul(pl.from_roots([0.0, -4.0]), [1.0, 4.0, 20.0]))
rl = st.root_locus(G, np.logspace(-1, 3, 1000))
print(f"rules: centroid {rl['centroid']:.2f}, asymptotes {rl['asymptote_angles']} deg, breakaway {np.round(rl['breakaway'], 3)}, departure angles {({k: round(float(v), 1) for k, v in rl['departure_angles'].items()})}")
# imaginary-axis crossing: Routh on s^4 + 8 s^3 + 36 s^2 + 80 s + K
Kc = None
for K in np.linspace(1, 400, 4000):
    if not st.routh(pl.add(G.den, K * G.num))["stable"]:
        Kc = K; break
print(f"Routh: unstable from K = {Kc:.0f} (closed form: 8*36*80 - 80^2 = 8^2 K -> K = {(8 * 36 * 80 - 80**2) / 64:.0f})")
cross = pl.roots(pl.add(G.den, (8 * 36 * 80 - 80**2) / 64 * G.num))
print("poles at that gain:", np.round(cross, 3))

rules: centroid -2.00, asymptotes [ 45. 135. 225. 315.] deg, breakaway [-2.], departure angles {(-2+4j): -90.0}
Routh: unstable from K = 260 (closed form: 8*36*80 - 80^2 = 8^2 K -> K = 260)
poles at that gain: [-4.+3.162j -4.-3.162j  0.+3.162j  0.-3.162j]


Four poles, no zeros: asymptotes at ±45° and ±135° from the centroid −2, a breakaway between 0 and −4, and
departure angles from −2 ± 4j that point towards the right half-plane. The Routh array gives the crossing gain
260 and the frequency ±j√10 where the locus meets the imaginary axis; the computed locus confirms both.

## Exercise 2

Add a zero to 1/(s (s + 1)(s + 2)) at -0.5, -1.5 and -3 in turn and describe how the locus and the maximum achievable damping change. Which position is best for a fast, well-damped loop?

In [3]:
for z in (-0.5, -1.5, -3.0):
    Gz = sy.tf(pl.from_roots([z]), pl.from_roots([0.0, -1.0, -2.0]))
    gains = np.logspace(-2, 3, 3000)
    rl = st.root_locus(Gz, gains)
    best = None
    for K, r in zip(gains, rl["branches"]):
        dom = r[np.argmax(r.real)]                                   # the dominant (slowest) pole
        if abs(dom.imag) > 1e-9 and dom.real <= -0.5:                # the first gain at which the dominant pair is at least as fast as sigma = 0.5
            best = (K, dom, -dom.real / abs(dom)); break
    print(f"zero at {z:4}: centroid {rl['centroid']:.2f}, always stable: {np.all(rl['branches'].real < 1e-9)}; " + (f"dominant pair reaches sigma = 0.5 at K = {best[0]:.2f} with damping ratio {best[2]:.3f}" if best else "the dominant pair never reaches sigma = 0.5 (the vertical asymptotes sit to its right)"))

zero at -0.5: centroid -1.25, always stable: True; the dominant pair never reaches sigma = 0.5 (the vertical asymptotes sit to its right)
zero at -1.5: centroid -0.75, always stable: True; dominant pair reaches sigma = 0.5 at K = 0.38 with damping ratio 0.998


zero at -3.0: centroid 0.00, always stable: True; the dominant pair never reaches sigma = 0.5 (the vertical asymptotes sit to its right)


A zero on the real axis pulls the locus towards it and, by reducing the relative degree to two, makes the
asymptotes vertical at the centroid: all three loops are stable at every gain, but they differ in what the
dominant pole does. The zero at −0.5 captures the branch leaving the origin: the slowest closed-loop pole is
real and creeps towards −0.5, so the loop can never be faster than σ = 0.5 whatever the gain. The zero at
−1.5 lets the branches from 0 and −1 meet, break away and reach σ = 0.5 as a well-damped pair - the useful
case. The zero at −3 moves the centroid to the origin, and the complex pair heads up the imaginary axis without
ever reaching σ = 0.5. A zero near the dominant poles, but not between the slowest pole and the origin, is
the one that helps.

## Exercise 3

**Implement it yourself:** compute a root locus by solving D(s) + K N(s) = 0 for 500 gains and matching each root to the nearest root at the previous gain (so that branches are continuous); plot it against `stability.root_locus`.

In [4]:
G = sy.tf([1.0], pl.from_roots([0.0, -1.0, -2.0]))
gains = np.logspace(-2, 2, 500)
prev = None; branches = []
for K in gains:
    r = pl.roots(pl.add(G.den, K * G.num))
    if prev is not None:
        order, avail = [], list(range(3))
        for pk in prev:
            j = min(avail, key=lambda j: abs(r[j] - pk)); order.append(j); avail.remove(j)
        r = r[order]
    branches.append(r); prev = r
branches = np.array(branches)
lib = st.root_locus(G, gains)["branches"]
print("branches matched to the library's at every gain:", f"{np.max([np.max(np.abs(np.sort_complex(a) - np.sort_complex(b))) for a, b in zip(branches, lib)]):.1e}")
print("continuity: largest jump between consecutive gains along a branch", f"{np.max(np.abs(np.diff(branches, axis=0))):.3f}")

branches matched to the library's at every gain: 0.0e+00
continuity: largest jump between consecutive gains along a branch 0.065


Solving the characteristic polynomial for each gain gives the points; matching each root to the nearest one at
the previous gain strings them into continuous branches (a greedy assignment that works when the gain steps
are small). The library does the same, which is why the branches, not just the point sets, agree.